In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType
from pyspark.sql.functions import col
from pyspark.sql import functions as F

spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA silver")

In [0]:
# Lecture streaming depuis Bronze
bronze_stream = (
    spark.readStream
         .table("main.bronze.transactions_bronze_stream")
)

bronze_stream.printSchema() 

In [0]:
silver_stream = (
    bronze_stream
    # Nettoyage des valeurs nulles
    .na.drop(subset=["Amount", "Class"])
    
    # Typage
    .withColumn("Amount", F.col("amount").cast("double"))
    .withColumn("is_fraud", F.col("Class").cast("int"))
    
    # Suppression des doublons
    .dropDuplicates()

    # Suppression de la colonne Class
    .drop("Class")

    # Renommer les colonnes
    .withColumnRenamed("Amount", "amount")
    .withColumnRenamed("Time", "time")

)

In [0]:
# Ecriture streaming vers la table Silver
silver_checkpoint = "/Volumes/main/silver/silver_volume/_checkpoints/silver_stream"

query = (
    silver_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", silver_checkpoint)
        .trigger(once=True)
        .table("transactions_silver_stream")
)

In [0]:
silver_stream.printSchema()